In [ ]:
import numpy as np
import json
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import urllib.request
import os


In [ ]:

# ==========================================
# 1. From Stroke-3 to Stroke-5
# ==========================================

def drawing_to_stroke5(drawing, max_len=200):
    '''
    Convert a Quick Draw drawing to stroke-5 format.
    Output shape: (seq_len + 1, 5)
    Columns: [dx, dy, p1, p2, p3]
    A special end-of-sequence token [0, 0, 0, 0, 1] is appended at the end.
    '''
    strokes = []
    for stroke in drawing:
        xs, ys = stroke[0], stroke[1]
        for i in range(len(xs)):
            dx = xs[i] - xs[i-1] if i > 0 else 0
            dy = ys[i] - ys[i-1] if i > 0 else 0
            if i < len(xs) - 1:
                p1, p2, p3 = 1, 0, 0   # pen on paper
            else:
                p1, p2, p3 = 0, 1, 0   # pen lifted
            strokes.append([dx, dy, p1, p2, p3])

    # Append end-of-sequence token
    strokes.append([0, 0, 0, 0, 1])

    s5 = np.array(strokes, dtype=np.float32)
    if len(s5) > max_len:
        s5 = s5[:max_len]
        s5[-1] = [0, 0, 0, 0, 1]   # force last step to be EOS
    return s5

# Download verification setup
os.makedirs('data', exist_ok=True)
path = 'data/cat.ndjson'
if not os.path.exists(path):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/quickdraw_dataset/full/simplified/cat.ndjson', path)

drawings = []
with open(path) as f:
    for line in f:
        drawings.append(json.loads(line))
        if len(drawings) >= 10: break

s5 = drawing_to_stroke5(drawings[0]['drawing'])
print('Stroke-5 shape:', s5.shape)
print('First 5 rows (dx, dy, p1, p2, p3):')
print(s5[:5])
print('Last row (EOS token):', s5[-1])

# TODO: Check that at every step exactly one of p1, p2, p3 is 1.
#       Hint: s5[:, 2:].sum(axis=1) should be all ones.
is_one_hot = np.all(s5[:, 2:].sum(axis=1) == 1.0)
print(f"Validation: Is the pen state strictly one-hot encoded? {is_one_hot}")

# TODO: Convert drawings[2] to stroke-5 and print how many steps have p2=1 (pen lifts).
#       What does this number correspond to visually?
s5_drawing2 = drawing_to_stroke5(drawings[2]['drawing'])
pen_lifts = int(s5_drawing2[:, 3].sum())
print(f"Number of steps where p2=1 (pen lifts): {pen_lifts}")

# OBSERVATIONS:
# 1. The number of steps where p2=1 corresponds exactly to the total number of independent lines 
#    or individual pen strokes that make up the doodle.
# 2. Visually, every time a user finishes drawing a continuous segment (like an ear, a whisker, 
#    or an eye) and lifts their pen up to move to the next part of the canvas, a p2=1 flag is stored.



In [ ]:

# ==========================================
# 2. Normalisation for Stroke-5
# ==========================================

def normalise_stroke5(stroke5):
    s = stroke5.copy()
    coords = s[:, :2]
    s[:, :2] = (coords - coords.mean(axis=0)) / (coords.std(axis=0) + 1e-8)
    return s

s5_norm = normalise_stroke5(s5)
print('dx/dy mean after normalisation:', s5_norm[:, :2].mean(axis=0).round(3))
print('dx/dy std  after normalisation:', s5_norm[:, :2].std(axis=0).round(3))
print('Pen states unchanged:', s5_norm[:, 2:].sum(axis=1)[:5])   # should all be 1



In [ ]:

# ==========================================
# 3. Updated Dataset
# =========================================

class QuickDrawDataset(Dataset):
    def __init__(self, path, max_len=200, max_samples=5000):
        self.samples = []
        with open(path) as f:
            for i, line in enumerate(f):
                if i >= max_samples: break
                d  = json.loads(line)
                s5 = drawing_to_stroke5(d['drawing'], max_len=max_len)
                s5 = normalise_stroke5(s5)
                self.samples.append(torch.tensor(s5, dtype=torch.float32))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def collate_fn(batch):
    lengths = [seq.shape[0] for seq in batch]
    padded  = pad_sequence(batch, batch_first=True, padding_value=0.0)
    return padded, lengths

dataset = QuickDrawDataset(path, max_len=200, max_samples=3000)
loader  = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

padded, lengths = next(iter(loader))
print('Batch shape:', padded.shape)   # (32, max_len_in_batch, 5)
print('Min length:', min(lengths), '  Max length:', max(lengths))

# TODO: Why does padding_value=0.0 work for stroke-5 but is slightly ambiguous?
#       Hint: what does [0, 0, 0, 0, 0] mean as a stroke? Is it a valid stroke-5 vector?
#       How could you make padding unambiguous?

# OBSERVATIONS:
# 1. Padding with 0.0 fills out short records with vectors of [0, 0, 0, 0, 0]. In stroke-5, 
#    a valid pen state must be one-hot encoded, meaning a row of all zeros [0, 0, 0] is invalid 
#    and does not represent drawing (p1), lifting (p2), or stopping (p3). 
# 2. While this invalid mask state technically makes the padding mathematically recognizable, 
#    it remains structurally ambiguous because it overloads the numeric zero values used for deltas (dx=0, dy=0).
# 3. To make padding completely unambiguous, we can enforce masks explicitly using PyTorch's 
#    `pack_padded_sequence` utility, which passes the actual sequence lengths vector directly to the LSTM. 
#    This guarantees the neural network completely ignores any numeric values sitting inside the padded steps.